# Notebook de alumno · PyTorch Lightning con CNN sobre MNIST

Este notebook está pensado para **practicar**.  
No está completamente cerrado: incluye zonas para que modifiques la arquitectura, el optimizador y varios hiperparámetros.

## Objetivos

- Entender la estructura básica de un proyecto con **Lightning**
- Completar y modificar una CNN
- Probar distintos optimizadores
- Comparar resultados
- Acostumbrarse a leer la documentación oficial

## Documentación recomendada

- **Documentación general de Lightning**: <https://lightning.ai/docs/pytorch/stable/index.html>  
- **Lightning in 15 minutes**: <https://lightning.ai/docs/pytorch/stable/starter/introduction.html>  
- **Basic skills / core skills**: <https://lightning.ai/docs/pytorch/stable/levels/core_skills.html>  

Según la documentación oficial, Lightning organiza el código PyTorch para reducir *boilerplate* y facilitar escalado, testing y entrenamiento en distintos dispositivos. citeturn0search0turn0search2turn0search10

## 1. Instalación

Si estás en Colab o en un entorno limpio, descomenta la siguiente línea.

In [ ]:
# !pip install lightning torchvision matplotlib

## 2. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

import matplotlib.pyplot as plt

## 3. Dispositivo disponible

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo disponible:", device)

## 4. Configuración del experimento

### Actividad
Prueba a cambiar algunos de estos valores y compara resultados.

In [ ]:
BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 8
NUM_WORKERS = 4
EARLY_STOPPING_PATIENCE = 3

## 5. Dataset MNIST

In [ ]:
transform = transforms.ToTensor()

train_dataset_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset_full,
    [train_size, val_size]
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

## 6. Visualización rápida

Comprueba qué aspecto tienen las imágenes.

In [ ]:
x0, y0 = train_dataset[0]
plt.imshow(x0.squeeze(), cmap="gray")
plt.title(f"Etiqueta: {y0}")
plt.axis("off")
plt.show()

## 7. DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

## 8. Modelo base

### Actividades sugeridas

1. Cambia el número de filtros de las convoluciones.  
2. Añade una tercera capa convolucional.  
3. Añade una capa `Dropout`.  
4. Cambia el número de neuronas de la capa densa.

En la documentación oficial, `LightningModule` es una de las piezas centrales del framework, junto con `Trainer`. citeturn0search0turn0search6

In [ ]:
class LitMNISTCNN(L.LightningModule):
    def __init__(self, lr=1e-3, optimizer_name="adam", dropout_p=0.0):
        super().__init__()
        self.save_hyperparameters()

        # ---- PARTE MODIFICABLE POR EL ALUMNO ----
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)  # opcional

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(p=dropout_p)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        # -----------------------------------------

    def forward(self, x):
        # ---- PARTE MODIFICABLE POR EL ALUMNO ----
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)

        # Si activas conv3, tendrás que ajustar aquí el flujo
        # y recalcular la entrada de fc1

        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        # -----------------------------------------
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        # ---- PARTE MODIFICABLE POR EL ALUMNO ----
        if self.hparams.optimizer_name == "adam":
            optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        elif self.hparams.optimizer_name == "sgd":
            optimizer = torch.optim.SGD(self.parameters(), lr=self.hparams.lr, momentum=0.9)
        elif self.hparams.optimizer_name == "rmsprop":
            optimizer = torch.optim.RMSprop(self.parameters(), lr=self.hparams.lr)
        else:
            raise ValueError(f"Optimizador no soportado: {self.hparams.optimizer_name}")
        # -----------------------------------------

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

## 9. Instanciación del modelo

### Actividad
Prueba varios optimizadores:

- `"adam"`
- `"sgd"`
- `"rmsprop"`

Y distintos valores de `dropout_p`.

In [ ]:
model = LitMNISTCNN(
    lr=LR,
    optimizer_name="adam",   # cambia aquí
    dropout_p=0.0            # cambia aquí
)

model

## 10. Callbacks y logger

En la documentación, Lightning explica que `Trainer` gestiona internamente detalles del loop, callbacks, dispositivos y dataloaders. citeturn0search6

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=EARLY_STOPPING_PATIENCE,
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-mnist-cnn-{epoch:02d}-{val_loss:.4f}"
)

logger = CSVLogger(save_dir="logs", name="mnist_cnn_lightning_student")

## 11. Trainer

### Actividad
Cambia `max_epochs` y observa el efecto.

In [ ]:
trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    logger=logger,
    callbacks=[early_stopping, checkpoint_callback]
)

## 12. Entrenamiento

In [ ]:
trainer.fit(model, train_loader, val_loader)

## 13. Evaluación final

In [ ]:
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

## 14. Ruta del mejor checkpoint

In [ ]:
print("Mejor checkpoint:")
print(checkpoint_callback.best_model_path)

## 15. Carga manual del mejor modelo

In [ ]:
best_model = LitMNISTCNN.load_from_checkpoint(checkpoint_callback.best_model_path)
best_model.eval()
print("Modelo cargado")

## 16. Predicción manual

### Actividad
Cambia el índice y prueba varias imágenes.

In [ ]:
idx = 0  # cambia este valor
x, y = test_dataset[idx]

with torch.no_grad():
    logits = best_model(x.unsqueeze(0))
    pred = torch.argmax(logits, dim=1).item()

plt.imshow(x.squeeze(), cmap="gray")
plt.title(f"Real: {y} | Predicción: {pred}")
plt.axis("off")
plt.show()

## 17. Registro de experimentos

Lightning puede registrar métricas automáticamente con `self.log(...)`, y su documentación incluye una guía específica para logging. citeturn0search13

Si quieres explorar los resultados, busca los archivos CSV generados en la carpeta `logs/`.

In [ ]:
import os

for root, dirs, files in os.walk("logs"):
    level = root.replace("logs", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files[:10]:
        print(f"{subindent}{f}")

## 18. Retos propuestos

### Reto 1
Añade una tercera convolución y ajusta correctamente la entrada a la primera capa densa.

### Reto 2
Usa `optimizer_name="sgd"` y compara con Adam.

### Reto 3
Añade `Dropout(0.3)` o `Dropout(0.5)` y observa si mejora la validación.

### Reto 4
Cambia `monitor="val_loss"` por `monitor="val_acc"` en el callback de checkpoint.

### Reto 5
Intenta superar un **98 % de accuracy** en test.

## 19. Preguntas de reflexión

1. ¿Qué partes del entrenamiento escribes tú y cuáles ejecuta Lightning por debajo?
2. ¿Qué ventajas tiene `EarlyStopping`?
3. ¿Qué diferencia hay entre cambiar la arquitectura y cambiar el optimizador?
4. ¿Qué experimento te ha dado mejor resultado?
5. ¿Cuándo preferirías usar PyTorch puro en lugar de Lightning?